https://github.com/ulab-uiuc/LLMRouter

In [1]:
import os

# Check and clone only if the repository doesn't already exist locally
if not os.path.exists("LLMRouter"):
    !git clone https://github.com/ulab-uiuc/LLMRouter.git

# Install the package in editable mode so Python can resolve 'llmrouter' imports
%pip install -e ./LLMRouter

Obtaining file:///D:/college%20stuff/sem7/minor%20%28%20pinnacle%29/EcoGpt%20-minor/research%20and%20experiments/Router/experiments/exp_routeLLM/LLMRouterNote: you may need to restart the kernel to use updated packages.

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for llmrouter-lib (pyproject.toml): started
  Building editable for llmrouter-lib (pyproject.toml): finished with status 'done'
  Created wheel for llmrouter-lib: filename=llmrouter_lib-0.4.0-0.editable-py3-none-any.whl size=20332 sha256=c9d2e8c63563f1ed1bf0aebcc

In [8]:
import json
import os

# Define your model candidates mapping specifying Ollama's local OpenAI-compatible endpoint
llm_candidates = {
    "strong_model": {
        "model_name": "gpt-oss:120b-cloud",
        "provider": "ollama",
        "api_endpoint": "http://localhost:11434/v1",
        "cost": 0.03,
        "capability": 0.95
    },
    "weak_model": {
        "model_name": "qwen2.5:3b",
        "provider": "ollama",
        "api_endpoint": "http://localhost:11434/v1",
        "cost": 0.002,
        "capability": 0.60
    }
}

# Save candidate settings to the configuration directory expected by LLMRouter
os.makedirs("LLMRouter/data/llm_candidates", exist_ok=True)
candidate_path = "LLMRouter/data/llm_candidates/default_llm.json"

with open(candidate_path, "w") as f:
    json.dump(llm_candidates, f, indent=4)

print(f"Candidate configuration successfully written to {candidate_path}")

Candidate configuration successfully written to LLMRouter/data/llm_candidates/default_llm.json


In [9]:
import os
import yaml
import torch.nn as nn
from llmrouter.models.meta_router import MetaRouter

# 1. Set environment API variable (Ollama requires a dummy string to pass client validation)
os.environ["API_KEYS"] = "ollama"

# 2. Create the router configuration YAML pointing to your JSON candidate file
config_path = "local_router_config.yaml"
config_content = {
    "data_path": {
        "llm_data": "LLMRouter/data/llm_candidates/default_llm.json"
    }
}

with open(config_path, "w") as f:
    yaml.dump(config_content, f)

# 3. Define a Custom Router using LLMRouter's MetaRouter base class
class LocalCostRouter(MetaRouter):
    def __init__(self, yaml_path: str):
        # Pass a identity layer if no trainable model weights are required
        super().__init__(model=nn.Identity(), yaml_path=yaml_path)

    def route_single(self, query_input: dict) -> dict:
        query = query_input["query"]
        
        # Simple routing rule: route long/complex queries to strong_model, otherwise weak_model
        if len(query) > 60 or "explain" in query.lower():
            selected_model = "strong_model"
        else:
            selected_model = "weak_model"
            
        return {
            "query": query,
            "model_name": selected_model,
            "model_info": self.llm_data[selected_model]
        }

# 4. Instantiate the Router
router = LocalCostRouter(yaml_path=config_path)

# 5. Run test queries
simple_test = router.route_single({"query": "Hi, what time is it?"})
print("Simple Query:")
print(f"  Target Model: {simple_test['model_name']}")
print(f"  API Endpoint: {simple_test['model_info']['api_endpoint']}\n")

complex_test = router.route_single({
    "query": "Explain quantum entanglement and write a simulation step in Python."
})
print("Complex Query:")
print(f"  Target Model: {complex_test['model_name']}")
print(f"  API Endpoint: {complex_test['model_info']['api_endpoint']}")

TypeError: Can't instantiate abstract class LocalCostRouter with abstract method route_batch

In [ ]:
import pandas as pd
question_df = pd.read_csv("Domain Agnostic Bulk Questions for Reasoning Vs Non Reasoning Classification(Domain Agnostic Questions).csv", encoding='latin-1')
question_df.head()

,Category,Question,Answer,Question Type
0,Credit Risk,A bank is evaluating two borrowers for a corpo...,Borrower B is likely to receive a better credi...,Reasoning
1,Equities,An investor buys shares of a company at ?850 p...,Capital gain = ?960 ? ?850 = ?110. Total gain ...,Reasoning
2,Banking,"A commercial bank has ?1,000 crore in deposits...","CRR requirement = 4.5% of ?1,000 crore = ?45 c...",Reasoning
3,Financial Risk Assessment,A portfolio manager holds a portfolio consisti...,Portfolio return = (0.60 × ?18%) + (0.40 × 5%)...,Reasoning
4,Corporate Finance,A company is deciding between financing a new ...,Advantages of debt financing include lower fin...,Reasoning


In [6]:
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# ============================================================
# CONFIG
# ============================================================

INPUT_CSV = "Domain Agnostic Bulk Questions for Reasoning Vs Non Reasoning Classification(Domain Agnostic Questions).csv"
OUTPUT_CSV = "questions_routed.csv"

THRESHOLD = 50

# ============================================================
# FAKE OPENAI-COMPATIBLE SERVER
# ============================================================

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy-key"
)

# ============================================================
# MODELS EXPOSED BY YOUR FAKE SERVER
# ============================================================

llm_candidates = {
    "strong_model": {
        "model_name": "big-model"
    },
    "weak_model": {
        "model_name": "small-model"
    }
}


# ============================================================
# ROUTER
# ============================================================

def route_and_generate(query: str, threshold: int = 50):

    # Simple routing logic
    if len(query) > threshold:
        model_key = "strong_model"
        model_name = llm_candidates["strong_model"]["model_name"]
    else:
        model_key = "weak_model"
        model_name = llm_candidates["weak_model"]["model_name"]

    print(
        f"--> [Router Decision]: "
        f"{model_key} ({model_name})"
    )

    # Send request to fake endpoint
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": query
            }
        ]
    )

    response_text = response.choices[0].message.content

    return response_text, model_key, model_name


# ============================================================
# LOAD QUESTIONS
# ============================================================

question_df = pd.read_csv(
    INPUT_CSV,
    encoding="latin-1"
)

print(f"Loaded {len(question_df)} questions")


# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Category",
    "Question",
    "Answer",
    "Question Type"
]

missing_columns = [
    col
    for col in required_columns
    if col not in question_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# CREATE OUTPUT COLUMNS
# ============================================================

question_df["model_used"] = None
question_df["router_decision"] = None
question_df["model_response"] = None


# ============================================================
# RUN ROUTER
# ============================================================

for idx, row in tqdm(
    question_df.iterrows(),
    total=len(question_df),
    desc="Routing questions"
):

    question_text = str(row["Question"])

    try:

        response_text, model_key, model_name = route_and_generate(
            question_text,
            threshold=THRESHOLD
        )

        # Which actual endpoint/model was used
        question_df.at[idx, "model_used"] = model_name

        # Router's logical decision
        question_df.at[idx, "router_decision"] = model_key

        # Response returned by fake endpoint
        question_df.at[idx, "model_response"] = response_text

    except Exception as e:

        print(f"\nError at row {idx}: {e}")

        question_df.at[idx, "model_used"] = "ERROR"
        question_df.at[idx, "router_decision"] = "ERROR"
        question_df.at[idx, "model_response"] = str(e)

    # Save after every row
    question_df.to_csv(
        OUTPUT_CSV,
        index=False
    )


# ============================================================
# SUMMARY
# ============================================================

print("\n========================================")
print("Routing complete")
print("========================================")

print(f"Input : {INPUT_CSV}")
print(f"Output: {OUTPUT_CSV}")

print("\nModel usage:")
print(question_df["model_used"].value_counts())

print("\nRouter decisions:")
print(question_df["router_decision"].value_counts())

print("\nResponse values:")
print(question_df["model_response"].value_counts())

Loaded 180 questions


Routing questions:   0%|          | 0/180 [00:00<?, ?it/s]

--> [Router Decision]: strong_model (big-model)


Routing questions:  19%|█▉        | 34/180 [00:02<00:05, 24.35it/s]

--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big

Routing questions:  28%|██▊       | 50/180 [00:02<00:03, 38.85it/s]

... (5 outputs left out)

Routing questions:  83%|████████▎ | 150/180 [00:02<00:00, 132.30it/s]

--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big

Routing questions: 100%|██████████| 180/180 [00:03<00:00, 56.73it/s] 

--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: weak_model (small-model)
--> [Router Decision]: strong_model (big-model)
--> [Router Decision]: strong_model (big-model)

Routing complete
Input : Domain Agnostic Bulk Questions for Reasoning Vs Non Reasoning Classification(Domain Agnostic Questions).csv
Output: questions_routed.csv

Model usage:
model_used
big-model      174
small-model      6
Name: count, dtype: int64

Router decisions:
router_decision
strong_model    174
weak_model        6
N